# Chunking pipeline for FoodScholar — Guides (Overlapping)

512‑token chunks with 64‑token overlap. Full sentences, chapter‑aware (no cross‑section overlap), tables included as Markdown, images ignored.

Uses `removed_pages_log.txt` to skip pre‑identified non‑content pages.

In [ ]:
# --- Environment: HuggingFace cache ---
import os
from pathlib import Path

CACHE_ROOT = Path(".cache")
HF_HOME_DIR = CACHE_ROOT / "huggingface"
HF_HUB_CACHE_DIR = HF_HOME_DIR / "hub"
HF_TRANSFORMERS_CACHE_DIR = HF_HOME_DIR / "transformers"

for path in [CACHE_ROOT, HF_HOME_DIR, HF_HUB_CACHE_DIR, HF_TRANSFORMERS_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ.update(
    {
        "XDG_CACHE_HOME": str(Path(CACHE_ROOT)),
        "HF_HOME": str(HF_HOME_DIR),
        "HF_HUB_CACHE": str(HF_HUB_CACHE_DIR),
        "HUGGINGFACE_HUB_CACHE": str(HF_HUB_CACHE_DIR),
        "TRANSFORMERS_CACHE": str(HF_TRANSFORMERS_CACHE_DIR),
        "HF_HUB_DISABLE_SYMLINKS_WARNING": "1",
    }
)

## 1. Parse removed pages log

In [ ]:
import re

REMOVED_PAGES_LOG = "./pdfs.filtered.txt"

# Parse:  filename: <name> | removed_pages: [p1, p2, ...]
_ENTRY_RE = re.compile(
    r"^filename:\s*(.+?)\s*\|\s*removed_pages:\s*\[([^\]]*)\]"
)


def load_excluded_pages(log_path: str = REMOVED_PAGES_LOG) -> dict[str, set[int]]:
    """Parse removed_pages_log.txt → {filename_base: {page_number, ...}}"""
    result = {}
    with open(log_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = _ENTRY_RE.match(line)
            if m:
                name = m.group(1).strip()
                pages_str = m.group(2).strip()
                pages = {int(p.strip()) for p in pages_str.split(",") if p.strip()}
                result[name] = pages
    return result


excluded_pages_map = load_excluded_pages()
print(f"✅ Loaded excluded pages for {len(excluded_pages_map)} guides")

# Show first few entries
for i, (name, pages) in enumerate(excluded_pages_map.items()):
    if i >= 5:
        print(f"   ... and {len(excluded_pages_map) - 5} more")
        break
    print(f"   {name}: {sorted(pages)}")

## 2. Configure PDF converter (images ignored)

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions # type: ignore
)

pdf_options = PdfPipelineOptions()
pdf_options.accelerator_options = AcceleratorOptions(
    device="cuda:1"
)
pdf_options.do_ocr = False
pdf_options.generate_picture_images = False  # don't extract picture assets

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
    }
)

print("✅ PDF converter ready (images ignored)")

## 3. Serializer, Tokenizer & Chunker (tables ✅, images ❌)

In [ ]:
from docling.chunking import HybridChunker # type: ignore
from docling_core.transforms.chunker.tokenizer.huggingface import (
    HuggingFaceTokenizer,
)
from transformers import AutoTokenizer
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
)
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer


# ---------------------------------------------------------------------------
# Custom serializer: tables as Markdown, images → empty string (ignored)
# ---------------------------------------------------------------------------
class MDTableNoImageSerializerProvider(ChunkingSerializerProvider):
    """Serializes tables as Markdown; ignores images completely."""

    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            table_serializer=MarkdownTableSerializer(),
        )


# ---------------------------------------------------------------------------
# Embedding tokenizer (used for token counting in chunks)
# ---------------------------------------------------------------------------
EMBED_MODEL_ID = "BAAI/bge-large-en-v1.5"

hf_tokenizer = AutoTokenizer.from_pretrained(
    EMBED_MODEL_ID,
    use_fast=False,
)

# ---------------------------------------------------------------------------
# Fine-grained HybridChunker (small max_tokens → small sentence-chunks)
# With max_tokens=80, chunks are ~40-80 tokens (1-3 sentences),
# giving precise ~64-token overlap control at the chunk level.
# ---------------------------------------------------------------------------
FINE_MAX_TOKENS = 80

hybrid_chunker = HybridChunker(
    tokenizer=HuggingFaceTokenizer(
        tokenizer=hf_tokenizer,
        max_tokens=FINE_MAX_TOKENS,
    ),
    serializer_provider=MDTableNoImageSerializerProvider(),
    merge_peers=True,
    repeat_table_header=False,
    always_emit_headings=False,
    omit_header_on_overflow=False,
)

print(f"✅ HybridChunker ready (fine max_tokens={FINE_MAX_TOKENS}, tables=md, images=off)")

## 4. Sliding-window overlap builder (512 tokens, 64 overlap, chapter‑aware)

In [ ]:
from itertools import groupby


def _get_heading_key(chunk):
    """Top-level heading — used to prevent cross-chapter overlap."""
    headings = chunk.meta.headings
    return headings[0] if headings else ""


def _count_tokens(text: str, tokenizer) -> int:
    return len(tokenizer.tokenize(text))


def _merge_batch(batch, tokenizer):
    """Merge a list of fine-chunks into a single result dict."""
    merged_text = " ".join(c.text.strip() for c in batch)
    page_numbers = sorted(set(
        prov.page_no
        for c in batch
        for item in getattr(c.meta, "doc_items", [])
        for prov in getattr(item, "prov", [])
        if getattr(prov, "page_no", None) is not None
    ))
    return {
        "text": merged_text,
        "page_numbers": page_numbers,
        "token_count": _count_tokens(merged_text, tokenizer),
        "num_sub_chunks": len(batch),
    }


def build_overlapping_chunks(
    fine_chunks,
    tokenizer,
    max_tokens: int = 512,
    overlap: int = 64,
):
    """
    Build final chunks of ~max_tokens with ~overlap tokens of overlap.

    Strategy:
      1. Group fine-chunks by top-level heading (no cross-section overlap).
      2. Within each heading group, slide a window over fine-chunks.
         Because FINE_MAX_TOKENS ≈ 80, fine-chunks are ~40-80 tokens,
         giving precise ~64-token overlap control.
      3. If all remaining text in a heading fits in one chunk,
         emit it as a single chunk (no further splitting).

    Guarantees:
      • No cross-section overlap (grouped by heading)
      • Full sentences (HybridChunker already respects boundaries)
      • Tables preserved as Markdown
      • Images absent
      • No redundant tail-only chunks

    Returns a list of dicts with keys:
      text, heading, page_numbers, token_count, num_sub_chunks
    """
    groups: list[tuple[str, list]] = []
    for heading, group in groupby(fine_chunks, key=_get_heading_key):
        groups.append((heading, list(group)))

    result: list[dict] = []

    for heading, group in groups:
        n = len(group)
        if n == 0:
            continue

        token_counts = [_count_tokens(c.text, tokenizer) for c in group]
        cumsum = [0] * (n + 1)
        for i in range(n):
            cumsum[i + 1] = cumsum[i] + token_counts[i]

        start = 0
        while start < n:
            # ── If all remaining fits in one chunk, emit and stop ──
            if cumsum[n] - cumsum[start] <= max_tokens:
                merged = _merge_batch(group[start:n], tokenizer)
                merged["heading"] = heading
                result.append(merged)
                break

            # ── Expand end as far as possible within max_tokens ──
            end = start
            while end < n and (cumsum[end + 1] - cumsum[start]) <= max_tokens:
                end += 1

            if end == start:
                batch = group[start:start + 1]
                start += 1
            else:
                batch = group[start:end]

                # ── Advance start for ~overlap tokens ──
                new_start = start
                for k in range(start + 1, end + 1):
                    tail = cumsum[end] - cumsum[k]
                    if tail >= overlap:
                        new_start = k
                    else:
                        break
                start = max(new_start, start + 1)

            merged = _merge_batch(batch, tokenizer)
            merged["heading"] = heading
            result.append(merged)

    return result


print("✅ build_overlapping_chunks() ready")

## 5. Process a single guide (convert → chunk → overlap → filter → csv)

In [ ]:
import os
import pandas as pd
import re as _re
import uuid


# Path to the guide PDF files and metadata CSV
GUIDES_PATH = "../data/original/guides/"
GUIDE_METADATA_CSV = os.path.join(GUIDES_PATH, "guide_metadata.csv")

OUTPUT_MAX_TOKENS = 512
OUTPUT_OVERLAP    = 64

PUNCT_ONLY_RE = _re.compile(r"^[\W_]+$", flags=_re.UNICODE)


guide_metadata_df = pd.read_csv(GUIDE_METADATA_CSV)
guide_metadata_map = (
    guide_metadata_df
    .assign(pdf_name=lambda df: df["pdf_name"].astype(str).str.strip())
    .set_index("pdf_name")
    .to_dict(orient="index")
)


def has_meaningful_text(text: str) -> bool:
    """True if text is non-empty and not purely punctuation."""
    s = str(text or "").strip()
    return bool(s) and not PUNCT_ONLY_RE.fullmatch(s)


def _get_guide_metadata(guide_filename: str, metadata_map: dict[str, dict]) -> dict[str, object]:
    """Return metadata row matched by the guide filename stem via pdf_name."""
    guide_name = os.path.splitext(os.path.basename(guide_filename))[0]
    metadata = metadata_map.get(guide_name)
    if metadata is None:
        raise KeyError(
            f"No metadata found in {GUIDE_METADATA_CSV} for pdf_name='{guide_name}'"
        )
    return {"pdf_name": guide_name, **metadata}


def _match_log_key(guide_filename: str, excluded_map: dict[str, set[int]]) -> tuple[str, set[int]]:
    """Match the guide PDF filename to the removed-pages log entry."""
    if guide_filename in excluded_map:
        return guide_filename, excluded_map[guide_filename]


    guide_stem = os.path.splitext(guide_filename)[0]
    for key, pages in excluded_map.items():
        if key == guide_stem or os.path.splitext(key)[0] == guide_stem:
            return key, pages


    return "", set()


def process_one_guide(
    guide_filename: str,
    excluded_map: dict[str, set[int]],
    label: str = "",
) -> pd.DataFrame:
    """
    Full pipeline for one guide PDF:
      1. Load guide metadata from guide_metadata.csv using pdf_name
      2. Match to removed_pages_log entry for excluded pages
      3. Convert PDF → DoclingDocument (images ignored)
      4. Fine-grained chunk with HybridChunker (max 80 tokens)
      5. Build overlapping chunks (512 tokens, 64 overlap, section-aware)
      6. Filter out excluded pages & meaningless text
      7. Return DataFrame with guide metadata
    """
    filepath = os.path.join(GUIDES_PATH, guide_filename)
    tag = label or guide_filename


    # ── 0. Load metadata ──
    guide_meta = _get_guide_metadata(guide_filename, guide_metadata_map)


    # Match to log
    matched_key, excluded_pages = _match_log_key(guide_filename, excluded_map)


    print(f"\n{'='*60}")
    print(f"📖 Processing: {tag}")
    print(f"   Path: {filepath}")
    print(
        f"   🌍 country={guide_meta['country']} | "
        f"📝 title={guide_meta['title']} | "
        f"👥 audience={guide_meta['audience']}"
    )
    if matched_key:
        print(f"   📋 Matched log key: {matched_key}")
        print(f"   🚫 Excluded pages: {sorted(excluded_pages)}")
    else:
        print(f"   ⚠️  No log entry found — including all pages")


    # ── 1. Convert PDF ──
    doc = converter.convert(source=filepath).document
    print(f"   ✅ PDF converted ({len(doc.texts)} text items)")


    # ── 2. Fine-grained chunking ──
    fine_chunks = list(hybrid_chunker.chunk(dl_doc=doc))
    print(f"   ✅ Fine chunks: {len(fine_chunks)}")


    # ── 3. Build overlapping chunks ──
    overlap_chunks = build_overlapping_chunks(
        fine_chunks,
        tokenizer=hf_tokenizer,
        max_tokens=OUTPUT_MAX_TOKENS,
        overlap=OUTPUT_OVERLAP,
    )
    print(f"   ✅ Overlap chunks: {len(overlap_chunks)}")


    # ── 4. Filter & build DataFrame ──
    rows = []
    for oc in overlap_chunks:
        if not has_meaningful_text(oc["text"]):
            continue
        if oc["page_numbers"] and any(p in excluded_pages for p in oc["page_numbers"]):
            continue


        rows.append({
            "chunk_id": str(uuid.uuid4()),
            "chunk_text": oc["text"],
            "type": "guide",
            "chunk_metadata": {
                "file": guide_filename,
                "urn": guide_meta["urn"],
                "pdf_name": guide_meta["pdf_name"],
                "country": guide_meta["country"],
                "title": guide_meta["title"],
                "audience": guide_meta["audience"],
                "pages": guide_meta["pages"],
                "heading": oc["heading"],
                "page_number": oc["page_numbers"][0] if oc["page_numbers"] else None,
            },
        })


    df = pd.DataFrame(rows)
    print(f"   ✅ Final rows (after filtering): {len(df)}")


    # Token-count stats
    if len(df) > 0:
        token_counts = [_count_tokens(t, hf_tokenizer) for t in df["chunk_text"]]
        print(f"   📊 Token stats — min: {min(token_counts)}, max: {max(token_counts)}, "
              f"mean: {sum(token_counts)/len(token_counts):.0f}")


    return df


print(f"✅ Loaded guide metadata for {len(guide_metadata_map)} guides")
print("✅ process_one_guide() ready")

## 6. Run on guide(s) & export CSVs

In [ ]:
# ── Guides to process ──
# Add guide PDF filenames from GUIDES_PATH here.
# Metadata is loaded from guide_metadata.csv via the pdf_name column.


GUIDES = [
    "be-dietary-recommendations-for-the-belgian-population.pdf",
    "bg-food-based-dietary-guidelines-for-adults-in-bulgaria.pdf",
    "de-eat-and-drink-well-recommendations-of-the-german-nutrition-society-dge.pdf",
    "dk-the-official-dietary-guidelines-good-for-health-and-climate.pdf",
    "es-healthy-and-sustainable-dietary-recommendations.pdf",
    "fi-sustainable-health-from-food.pdf",
    "fr-recommendations-relating-to-diet-physical-activity-and-a-sedentary-lifestyle.pdf",
    "hu-dietary-guidelines-for-4-17-year-olds.pdf",
    "hu-dietary-guidelines-for-the-general-population.pdf",
    "ie-cereals-shelf-fact-sheet.pdf",
    "ie-childrens-food-pyramid-poster-for-parents.pdf",
    "ie-childrens-food-pyramid-poster-for-professionals.pdf",
    "ie-childrens-food-pyramid-questions-and-answers.pdf",
    "ie-dairy-shelf-fact-sheet.pdf",
    "ie-expanded-food-pyramid-poster.pdf",
    "ie-fats-shelf-fact-sheet.pdf",
    "ie-food-guide-for-fats-oils-and-spreads.pdf",
    "ie-food-guide-on-milk-yogurt-and-cheese.pdf",
    "ie-food-guide-on-poultry-fish-eggs-beans-and-nuts.pdf",
    "ie-food-guide-on-vegatables-salad-and-fruit.pdf",
    "ie-food-guide-to-cereals-breads-potatoes-pasta-and-rice.pdf",
    "ie-food-pyramid-information-leaflet.pdf",
    "ie-food-pyramid-poster.pdf",
    "ie-food-pyramid-to-daily-meal-plan-for-aged-10.pdf",
    "ie-food-pyramid-to-daily-meal-plan-for-aged-21.pdf",
    "ie-food-pyramid-to-daily-meal-plan-for-aged-30.pdf",
    "ie-food-pyramid-to-daily-meal-plan-for-aged-52.pdf",
    "ie-guide-for-foods-high-in-fats-sugar-and-salt.pdf",
    "ie-happy-healthy-mealtimes.pdf",
    "ie-healthy-eating-guidelines-for-1-to-4-year-olds.pdf",
    "ie-key-messages.pdf",
    "ie-meal-plan-age-1.pdf",
    "ie-meal-plan-age-2.pdf",
    "ie-meal-plan-age-3.pdf",
    "ie-meal-plan-age-3-vegetarian.pdf",
    "ie-meal-plan-age-4.pdf",
    "ie-meat-shelf-fact-sheet.pdf",
    "ie-portions.pdf",
    "ie-snacks.pdf",
    "ie-top-shelf-fact-sheet.pdf",
    "ie-vegetable-shelf-fact-sheet.pdf",
    "ie-vitamin-d.pdf",
    "ie-the-children-s-food-pyramid.pdf",
    "mt-dietary-guidelines-for-maltese-adults.pdf",
    "nl-dutch-dietary-guidelines.pdf",
    "nl-eating-more-sustainably.pdf",
    "nl-the-wheel-of-five.pdf",
    "se-advice-about-food-for-you-who-are-breastfeeding.pdf",
    "se-advice-about-food-for-you-who-are-pregnant.pdf",
    "se-dietary-advice-for-adults.pdf",
    "se-good-food-for-children-between-one-and-two-years.pdf",
]


OUTPUT_DIR = "../data/chunks/guides"
os.makedirs(OUTPUT_DIR, exist_ok=True)
all_dfs = {}


for guide_filename in GUIDES:
    label = os.path.splitext(guide_filename)[0]
    df = process_one_guide(guide_filename, excluded_pages_map, label=label)
    all_dfs[label] = df


    # Save CSV
    csv_path = os.path.join(OUTPUT_DIR, f"chunks_guide_{label}.csv")
    df.to_csv(csv_path, index=False)
    print(f"   💾 Saved → {csv_path}")


# ── Summary ──
print(f"\n{'='*60}")
print("📊 ALL DONE — Summary")
total = sum(len(df) for df in all_dfs.values())
for label, df in all_dfs.items():
    print(f"   {label}: {len(df)} chunks")
print(f"   TOTAL: {total} chunks across {len(all_dfs)} guides")